# Men's Accessories Store – Backend Portal

**PF Lab Assignment 3**

A simple console-based Python store using CSV files to save products and orders.

Run the cells from top to bottom. The program creates the CSV files if they do not exist.

Sample admin login: `admin` / `1234`


## 1. Importing CSV and Preparing Files
Import the CSV module and prepare files for product and order information.

In [1]:
import csv

product_file = "products.csv"
order_file = "orders.csv"

product_fields = ["id", "name", "category", "price", "stock"]
order_fields = ["order_id", "customer", "product", "quantity", "total"]


def prepare_file(file_name, fields):
    try:
        file = open(file_name, "r")
        file.close()
    except FileNotFoundError:
        try:
            file = open(file_name, "w", newline="")
            writer = csv.DictWriter(file, fieldnames=fields)
            writer.writeheader()
            file.close()
        except PermissionError:
            print("Cannot create:", file_name)


prepare_file(product_file, product_fields)
prepare_file(order_file, order_fields)


## 2. Reading and Saving Records
Read saved information from CSV files and save updated information back into them.

In [2]:
def read_records(file_name):
    records = []
    file = None

    try:
        file = open(file_name, "r", newline="")
        reader = csv.DictReader(file)
        for row in reader:
            records.append(row)
    except FileNotFoundError:
        print("File not found:", file_name)
    except PermissionError:
        print("Cannot open:", file_name)
    finally:
        if file is not None:
            file.close()

    return records


def save_records(file_name, fields, records):
    file = None

    try:
        file = open(file_name, "w", newline="")
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(records)
    except PermissionError:
        print("Cannot save:", file_name)
    finally:
        if file is not None:
            file.close()


## 3. Starting Products
Add sample products only when the product file is empty. You can replace these examples.

In [3]:
products = read_records(product_file)
orders = read_records(order_file)

if len(products) == 0:
    products = [
        {"id": "101", "name": "Wrist Watch", "category": "Watch", "price": "2500", "stock": "5"},
        {"id": "102", "name": "Leather Wallet", "category": "Wallet", "price": "1200", "stock": "8"},
        {"id": "103", "name": "Sunglasses", "category": "Eyewear", "price": "1800", "stock": "6"},
        {"id": "104", "name": "Leather Belt", "category": "Belt", "price": "900", "stock": "10"}
    ]
    save_records(product_file, product_fields, products)


## 4. Admin Login
Check the username and password. The function returns True when the login is correct.

In [4]:
def user_login():
    print("\n--- Admin Login ---")
    username = input("Username: ")
    password = input("Password: ")

    if username == "admin" and password == "1234":
        print("Login successful.")
        return True
    else:
        print("Incorrect username or password.")
        return False


## 5. Product Functions
View, search, add and update products.

In [5]:
def view_products(products):
    print("\n--- Products ---")
    if len(products) == 0:
        print("No products available.")
    else:
        for product in products:
            print(product["id"], product["name"], product["category"],
                  "Price:", product["price"], "Stock:", product["stock"])


def search_product(products):
    name = input("Enter product name: ").lower()
    found = False

    for product in products:
        if name in product["name"].lower():
            print(product["id"], product["name"], "Price:", product["price"],
                  "Stock:", product["stock"])
            found = True

    if found == False:
        print("Product not found.")

    return found


def add_product(products):
    product_id = input("Enter product ID: ")

    for product in products:
        if product["id"] == product_id:
            print("This ID is already used.")
            return products

    name = input("Enter product name: ")
    category = input("Enter category: ")

    try:
        price = float(input("Enter price: "))
        stock = int(input("Enter stock quantity: "))

        if price <= 0 or stock < 0:
            print("Price must be above zero and stock cannot be negative.")
            return products

    except ValueError:
        print("Enter a valid number for price and stock.")
        return products

    new_product = {
        "id": product_id,
        "name": name,
        "category": category,
        "price": str(price),
        "stock": str(stock)
    }
    products.append(new_product)
    print("Product added.")
    return products


def update_product(products):
    product_id = input("Enter product ID: ")

    for product in products:
        if product["id"] == product_id:
            new_price = input("New price (Enter to keep old): ")
            new_stock = input("New stock (Enter to keep old): ")

            try:
                if new_price != "":
                    price = float(new_price)
                    if price > 0:
                        product["price"] = str(price)
                    else:
                        print("Price must be above zero.")

                if new_stock != "":
                    stock = int(new_stock)
                    if stock >= 0:
                        product["stock"] = str(stock)
                    else:
                        print("Stock cannot be negative.")

            except ValueError:
                print("Invalid number.")

            print("Product information checked.")
            return products

    print("Product not found.")
    return products


## 6. Place an Order
Check the product and available stock, calculate the bill and reduce the stock.

In [6]:
def place_order(products, orders):
    product_id = input("Enter product ID: ")
    selected_product = None

    for product in products:
        if product["id"] == product_id:
            selected_product = product
            break

    if selected_product is None:
        print("Product not found.")
        return products, orders

    try:
        quantity = int(input("Enter quantity: "))
        if quantity <= 0:
            print("Quantity must be greater than zero.")
            return products, orders

        stock = int(selected_product["stock"])
        if quantity > stock:
            print("Not enough stock available.")
            return products, orders

    except ValueError:
        print("Enter a valid whole number.")
        return products, orders

    customer = input("Enter customer name: ")
    total = float(selected_product["price"]) * quantity

    new_order = {
        "order_id": str(len(orders) + 1),
        "customer": customer,
        "product": selected_product["name"],
        "quantity": str(quantity),
        "total": str(total)
    }

    orders.append(new_order)
    selected_product["stock"] = str(stock - quantity)

    print("Order placed successfully.")
    print("Total bill: Rs.", total)
    return products, orders


## 7. View Orders
Display the orders stored in the order list.

In [7]:
def view_orders(orders):
    print("\n--- Order History ---")
    if len(orders) == 0:
        print("No orders found.")
    else:
        for order in orders:
            print("Order ID:", order["order_id"],
                  "| Customer:", order["customer"],
                  "| Product:", order["product"],
                  "| Quantity:", order["quantity"],
                  "| Total: Rs.", order["total"])


## 8. Sales Report
Add the total of each order and count the number of items sold.

In [8]:
def sales_report(orders):
    total_sales = 0
    items_sold = 0

    for order in orders:
        total_sales = total_sales + float(order["total"])
        items_sold = items_sold + int(order["quantity"])

    print("\n--- Sales Report ---")
    print("Total orders:", len(orders))
    print("Items sold:", items_sold)
    print("Total sales: Rs.", total_sales)
    return total_sales


## 9. Main Menu and Exit
The while loop keeps the menu running. Select 8 to save the data and exit.

In [9]:
def main():
    products = read_records(product_file)
    orders = read_records(order_file)

    if user_login() == False:
        print("Portal closed.")
        return

    while True:
        print("\n===== MEN'S ACCESSORIES STORE =====")
        print("1. View Products")
        print("2. Search Product")
        print("3. Add Product")
        print("4. Update Product")
        print("5. Place Order")
        print("6. View Orders")
        print("7. Sales Report")
        print("8. Exit")

        choice = input("Choose an option: ")

        if choice == "1":
            view_products(products)
        elif choice == "2":
            search_product(products)
        elif choice == "3":
            products = add_product(products)
        elif choice == "4":
            products = update_product(products)
        elif choice == "5":
            products, orders = place_order(products, orders)
        elif choice == "6":
            view_orders(orders)
        elif choice == "7":
            sales_report(orders)
        elif choice == "8":
            save_records(product_file, product_fields, products)
            save_records(order_file, order_fields, orders)
            print("Data saved. Thank you.")
            break
        else:
            print("Invalid choice. Select from 1 to 8.")

        save_records(product_file, product_fields, products)
        save_records(order_file, order_fields, orders)


main()



--- Admin Login ---
Username: admin
Password: 1234
Login successful.

===== MEN'S ACCESSORIES STORE =====
1. View Products
2. Search Product
3. Add Product
4. Update Product
5. Place Order
6. View Orders
7. Sales Report
8. Exit
Choose an option: 1

--- Products ---
101 Wrist Watch Watch Price: 2500 Stock: 5
102 Leather Wallet Wallet Price: 1200 Stock: 8
103 Sunglasses Eyewear Price: 1800 Stock: 6
104 Leather Belt Belt Price: 900 Stock: 10

===== MEN'S ACCESSORIES STORE =====
1. View Products
2. Search Product
3. Add Product
4. Update Product
5. Place Order
6. View Orders
7. Sales Report
8. Exit
Choose an option: 5
Enter product ID: 101
Enter quantity: 1
Enter customer name: daniyal
Order placed successfully.
Total bill: Rs. 2500.0

===== MEN'S ACCESSORIES STORE =====
1. View Products
2. Search Product
3. Add Product
4. Update Product
5. Place Order
6. View Orders
7. Sales Report
8. Exit
Choose an option: 7

--- Sales Report ---
Total orders: 1
Items sold: 1
Total sales: Rs. 2500.0

==